English: Explain environment setup for loading .env and setting JAVA_HOME / SPARK_HOME.
### Import environment

In [ ]:
# Load environment variables from .env and set JAVA_HOME and SPARK_HOME for Spark
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["JAVA_HOME"] = os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = os.getenv("SPARK_HOME")


### Import necessary libraries

In [ ]:
# Import SparkSession and functions alias for dataframe operations
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

### Create session Spark

In [ ]:
# Build or get a SparkSession running locally on all available cores
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("FootballES")
    .getOrCreate()
)

### Verify my session spark

In [ ]:
# Print the SparkSession object to confirm successful creation
print(spark)

### Analyze Data with Spark

In [281]:
# Read the transformed CSV file into a Spark DataFrame with header parsing
df = spark.read.format("csv").options(
    header="true").load("transformed_data.csv")

In [282]:
# Display the first 5 rows of the DataFrame for quick inspection
df.show(5)

+-----+----------+------------+--------------------+---------+---------+
|Round|      Date|      Team 1|              Team 2|FT Team 1|FT Team 2|
+-----+----------+------------+--------------------+---------+---------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|        0|        0|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|        2|        0|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|        0|        2|
|    1|2020-09-13|FC Barcelona|            Elche CF|     NULL|     NULL|
|    1|2020-09-13| Real Madrid|           Getafe CF|     NULL|     NULL|
+-----+----------+------------+--------------------+---------+---------+
only showing top 5 rows


In [283]:
# Create explicit aliases for team and score columns to standardize names
df = df.selectExpr(
    "*",
    "`Team 1` as `HomeTeam`",
    "`Team 2` as `AwayTeam`",
    "`FT Team 1` as `HomeTeamGoals`",
    "`FT Team 2` as `AwayTeamGoals`"
)

In [284]:
# Show a sample after renaming to verify aliases are correct
df.show(5)

+-----+----------+------------+--------------------+---------+---------+------------+--------------------+-------------+-------------+
|Round|      Date|      Team 1|              Team 2|FT Team 1|FT Team 2|    HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|
+-----+----------+------------+--------------------+---------+---------+------------+--------------------+-------------+-------------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|        0|        0|    SD Eibar|       RC Celta Vigo|            0|            0|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|        2|        0|  Granada CF|Athletic Club Bilbao|            2|            0|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|        0|        2|    Cádiz CF|          CA Osasuna|            0|            2|
|    1|2020-09-13|FC Barcelona|            Elche CF|     NULL|     NULL|FC Barcelona|            Elche CF|         NULL|         NULL|
|    1|2020-09-13| Real Madrid|           Getafe CF|   

In [285]:
# Drop original columns that are no longer needed after aliasing
df = df.drop("FT Team 1", "FT Team 2", "Team 1", "Team 2")

In [286]:
# Quick verify of DataFrame structure and values after dropping columns
df.show(5)

+-----+----------+------------+--------------------+-------------+-------------+
|Round|      Date|    HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|
+-----+----------+------------+--------------------+-------------+-------------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|            0|            0|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|            2|            0|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|            0|            2|
|    1|2020-09-13|FC Barcelona|            Elche CF|         NULL|         NULL|
|    1|2020-09-13| Real Madrid|           Getafe CF|         NULL|         NULL|
+-----+----------+------------+--------------------+-------------+-------------+
only showing top 5 rows


In [287]:
# Compute match result using numeric comparison of Home vs Away goals
df = df.withColumn("Results",
                   F.when(F.col("HomeTeamGoals") > F.col(
                       "AwayTeamGoals"), "HomeTeamWin")
                   .when(F.col("HomeTeamGoals") < F.col("AwayTeamGoals"), "AwayTeamWin")
                   .otherwise("Draw"))

In [288]:
# Show sample rows including the computed Results column
df.show(5)

+-----+----------+------------+--------------------+-------------+-------------+-----------+
|Round|      Date|    HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|
+-----+----------+------------+--------------------+-------------+-------------+-----------+
|    1|2020-09-12|    SD Eibar|       RC Celta Vigo|            0|            0|       Draw|
|    1|2020-09-12|  Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|
|    1|2020-09-12|    Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|
|    1|2020-09-13|FC Barcelona|            Elche CF|         NULL|         NULL|       Draw|
|    1|2020-09-13| Real Madrid|           Getafe CF|         NULL|         NULL|       Draw|
+-----+----------+------------+--------------------+-------------+-------------+-----------+
only showing top 5 rows


In [301]:
# Compute season start year from the match Date.
# If month >= 7 (July), season starts that calendar year (e.g. Aug 2013 -> 2013-14),
# otherwise the season started the previous year (e.g. May 2013 -> 2012-13).
df = df.withColumn("Date", F.col("Date").cast("date"))
df_result = df.withColumn("Season",
                          F.when(F.month(F.col("Date")) >= 8,
                                 F.concat(F.year(F.col("Date")), F.lit("/"), (F.year(F.col("Date")) + 1)))
                          .otherwise(
                              F.concat((F.year(F.col("Date")) - 1), F.lit("/"), F.year(F.col("Date"))))
                          )

In [302]:
# Show a few rows to verify Season extraction
df_result.limit(10).show()

+-----+----------+------------------+--------------------+-------------+-------------+-----------+---------+
|Round|      Date|          HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|   Season|
+-----+----------+------------------+--------------------+-------------+-------------+-----------+---------+
|    1|2020-09-12|          SD Eibar|       RC Celta Vigo|            0|            0|       Draw|2020/2021|
|    1|2020-09-12|        Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|2020/2021|
|    1|2020-09-12|          Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|2020/2021|
|    1|2020-09-13|      FC Barcelona|            Elche CF|         NULL|         NULL|       Draw|2020/2021|
|    1|2020-09-13|       Real Madrid|           Getafe CF|         NULL|         NULL|       Draw|2020/2021|
|    1|2020-09-13|  Deportivo Alavés|          Real Betis|            0|            1|AwayTeamWin|2020/2021|
|    1|2020-09-13|R

In [303]:
# Remove rows where goal values are missing to avoid cast/aggregation errors later
df_result = df_result.dropna(subset=["HomeTeamGoals", "AwayTeamGoals"])

In [304]:
# Verify rows after dropping nulls
df_result.show(5)

+-----+----------+------------------+--------------------+-------------+-------------+-----------+---------+
|Round|      Date|          HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|   Season|
+-----+----------+------------------+--------------------+-------------+-------------+-----------+---------+
|    1|2020-09-12|          SD Eibar|       RC Celta Vigo|            0|            0|       Draw|2020/2021|
|    1|2020-09-12|        Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|2020/2021|
|    1|2020-09-12|          Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|2020/2021|
|    1|2020-09-13|  Deportivo Alavés|          Real Betis|            0|            1|AwayTeamWin|2020/2021|
|    1|2020-09-13|Real Valladolid CF|       Real Sociedad|            1|            1|       Draw|2020/2021|
+-----+----------+------------------+--------------------+-------------+-------------+-----------+---------+
only showing top 5 

In [305]:
# Create indicator columns for wins/ties to use in group aggregations later
df_result = df_result.withColumn("HomeTeamWin", F.when(F.col("Results") == "HomeTeamWin", 1).otherwise(0)) \
    .withColumn("AwayTeamWin", F.when(F.col("Results") == "AwayTeamWin", 1).otherwise(0)) \
    .withColumn("GameTie", F.when(F.col("Results") == "Draw", 1).otherwise(0))

In [306]:
# Display a small sample of the DataFrame including new indicator columns
df_result.limit(10).show()

+-----+----------+------------------+--------------------+-------------+-------------+-----------+---------+-----------+-----------+-------+
|Round|      Date|          HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|   Season|HomeTeamWin|AwayTeamWin|GameTie|
+-----+----------+------------------+--------------------+-------------+-------------+-----------+---------+-----------+-----------+-------+
|    1|2020-09-12|          SD Eibar|       RC Celta Vigo|            0|            0|       Draw|2020/2021|          0|          0|      1|
|    1|2020-09-12|        Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|2020/2021|          1|          0|      0|
|    1|2020-09-12|          Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|2020/2021|          0|          1|      0|
|    1|2020-09-13|  Deportivo Alavés|          Real Betis|            0|            1|AwayTeamWin|2020/2021|          0|          1|      0|
|    1|2020-0

In [307]:
# Print schema to confirm data types of columns before casting
df_result.printSchema()

root
 |-- Round: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- HomeTeam: string (nullable = true)
 |-- AwayTeam: string (nullable = true)
 |-- HomeTeamGoals: string (nullable = true)
 |-- AwayTeamGoals: string (nullable = true)
 |-- Results: string (nullable = false)
 |-- Season: string (nullable = true)
 |-- HomeTeamWin: integer (nullable = false)
 |-- AwayTeamWin: integer (nullable = false)
 |-- GameTie: integer (nullable = false)



In [312]:
# Types: caster les colonnes en place sans créer de nouvelles colonnes
df_result = df_result.withColumn("HomeTeamGoals", F.col("HomeTeamGoals").cast("int")) \
    .withColumn("AwayTeamGoals", F.col("AwayTeamGoals").cast("int"))

In [311]:
# Show a sample after casting to check types and values
df_result.limit(10).show()

25/10/14 01:42:03 ERROR Executor: Exception in task 0.0 in stage 192.0 (TID 176)
org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '2020/2021' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 5 in cell [310]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:145)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.withException(UTF8StringUtils.scala:51)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.toIntExact(UTF8StringUtils.scala:34)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils.toIntExact(UTF8StringUtils.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.executio

NumberFormatException: [CAST_INVALID_INPUT] The value '2020/2021' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 5 in cell [310]


In [300]:
# Print final schema to ensure casting succeeded
df_result.printSchema()

root
 |-- Round: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- HomeTeam: string (nullable = true)
 |-- AwayTeam: string (nullable = true)
 |-- HomeTeamGoals: integer (nullable = true)
 |-- AwayTeamGoals: integer (nullable = true)
 |-- Results: string (nullable = false)
 |-- Season: integer (nullable = true)
 |-- HomeTeamWin: integer (nullable = false)
 |-- AwayTeamWin: integer (nullable = false)
 |-- GameTie: integer (nullable = false)



In [ ]:
# Drop unnecessary columns 'Date' and 'Round'
df = df.drop("Date", "Round")

In [ ]:
# Verify DataFrame after dropping unneeded columns
df.show(5)

+------------------+--------------------+-------------+-------------+-----------+------+-----------+-----------+-------+
|          HomeTeam|            AwayTeam|HomeTeamGoals|AwayTeamGoals|    Results|Season|HomeTeamWin|AwayTeamWin|GameTie|
+------------------+--------------------+-------------+-------------+-----------+------+-----------+-----------+-------+
|          SD Eibar|       RC Celta Vigo|            0|            0|       Draw|  2020|          0|          0|      1|
|        Granada CF|Athletic Club Bilbao|            2|            0|HomeTeamWin|  2020|          1|          0|      0|
|          Cádiz CF|          CA Osasuna|            0|            2|AwayTeamWin|  2020|          0|          1|      0|
|  Deportivo Alavés|          Real Betis|            0|            1|AwayTeamWin|  2020|          0|          1|      0|
|Real Valladolid CF|       Real Sociedad|            1|            1|       Draw|  2020|          0|          0|      1|
+------------------+------------

In [ ]:
# Aggregate statistics for home matches using Spark functions
df_home_matches = df.groupBy('Season', 'HomeTeam') \
                    .agg(F.sum('HomeTeamWin').alias('TotalHomeWin'),
                         F.sum('AwayTeamWin').alias('TotalHomeLoss'),
                         F.sum('GameTie').alias('TotalHomeTie'),
                         F.sum('HomeTeamGoals').alias('HomeScoredGoals'),
                         F.sum('AwayTeamGoals').alias('HomeAgainstGoals')
                         ).withColumnRenamed('HomeTeam', 'Team')

In [ ]:
# Show aggregated home match statistics for review
df_home_matches.show()

+------+--------------------+------------+-------------+------------+---------------+----------------+
|Season|                Team|TotalHomeWin|TotalHomeLoss|TotalHomeTie|HomeScoredGoals|HomeAgainstGoals|
+------+--------------------+------------+-------------+------------+---------------+----------------+
|  2016|Athletic Club Bilbao|          12|            3|           4|             36|              19|
|  2017|       Villarreal CF|           9|            5|           4|             32|              19|
|  2015|          Levante UD|           5|            8|           7|             21|              26|
|  2014|          Granada CF|           6|            7|           5|             14|              18|
|  2017|       RC Celta Vigo|           7|            8|           4|             26|              27|
|  2018|        FC Barcelona|          15|            1|           4|             57|              19|
|  2014|          Levante UD|           7|            7|           5|    

In [ ]:
# Aggregate statistics for away matches similarly and rename AwayTeam to Team for joining
df_away_matches = df.groupBy('Season', 'AwayTeam') \
                    .agg(F.sum('AwayTeamWin').alias('TotalAwayWin'),
                         F.sum('HomeTeamWin').alias('TotalAwayLoss'),
                         F.sum('GameTie').alias('TotalAwayTie'),
                         F.sum('AwayTeamGoals').alias('AwayScoredGoals'),
                         F.sum('HomeTeamGoals').alias('AwayAgainstGoals')
                         ).withColumnRenamed('AwayTeam', 'Team')

In [ ]:
# Display a few aggregated away match rows
df_away_matches.limit(10).show()

+------+--------------------+------------+-------------+------------+---------------+----------------+
|Season|                Team|TotalAwayWin|TotalAwayLoss|TotalAwayTie|AwayScoredGoals|AwayAgainstGoals|
+------+--------------------+------------+-------------+------------+---------------+----------------+
|  2016|Athletic Club Bilbao|           6|            9|           3|             19|              27|
|  2017|       Villarreal CF|          10|            7|           4|             23|              23|
|  2015|          Levante UD|           3|           14|           2|             13|              41|
|  2014|          Granada CF|           2|           12|           5|             13|              42|
|  2017|       RC Celta Vigo|           6|           12|           2|             32|              36|
|  2018|        FC Barcelona|          10|            2|           6|             45|              22|
|  2014|          Levante UD|           3|            7|           8|    

In [ ]:
# Join home and away aggregated stats on Season and Team to build team-level stats
df_team_stats = df_home_matches.join(
    df_away_matches, ['Season', 'Team'], 'inner')

In [244]:
df_team_stats.show()

+------+--------------------+------------+-------------+------------+---------------+----------------+------------+-------------+------------+---------------+----------------+
|Season|                Team|TotalHomeWin|TotalHomeLoss|TotalHomeTie|HomeScoredGoals|HomeAgainstGoals|TotalAwayWin|TotalAwayLoss|TotalAwayTie|AwayScoredGoals|AwayAgainstGoals|
+------+--------------------+------------+-------------+------------+---------------+----------------+------------+-------------+------------+---------------+----------------+
|  2016|Athletic Club Bilbao|          12|            3|           4|             36|              19|           6|            9|           3|             19|              27|
|  2017|       Villarreal CF|           9|            5|           4|             32|              19|          10|            7|           4|             23|              23|
|  2015|          Levante UD|           5|            8|           7|             21|              26|           3|     

In [246]:
df_team_stats.toPandas()

,Season,Team,TotalHomeWin,TotalHomeLoss,TotalHomeTie,HomeScoredGoals,HomeAgainstGoals,TotalAwayWin,TotalAwayLoss,TotalAwayTie,AwayScoredGoals,AwayAgainstGoals
0,2016,Athletic Club Bilbao,12,3,4,36,19,6,9,3,19,27
1,2017,Villarreal CF,9,5,4,32,19,10,7,4,23,23
2,2015,Levante UD,5,8,7,21,26,3,14,2,13,41
3,2014,Granada CF,6,7,5,14,18,2,12,5,13,42
4,2017,RC Celta Vigo,7,8,4,26,27,6,12,2,32,36
...,...,...,...,...,...,...,...,...,...,...,...,...
199,2019,Granada CF,5,3,1,11,6,2,5,2,13,19
200,2012,RC Celta Vigo,3,2,3,9,6,1,8,0,7,16
201,2018,SD Huesca,0,4,4,6,11,1,7,1,9,23
202,2013,Atlético Madrid,14,3,2,47,12,11,3,5,27,13


In [ ]:
# Create columns for total wins, losses, ties, scored goals, and against goals
df_team_stats_totals = df_team_stats.withColumn('TotalWins', F.col('TotalHomeWin') + F.col('TotalAwayWin')) \
    .withColumn('TotalLosses', F.col('TotalHomeLoss') + F.col('TotalAwayLoss')) \
    .withColumn('TotalTies', F.col('TotalHomeTie') + F.col('TotalAwayTie')) \
    .withColumn('TotalScoredGoals', F.col('HomeScoredGoals') + F.col('AwayScoredGoals')) \
    .withColumn('TotalAgainstGoals', F.col('HomeAgainstGoals') + F.col('AwayAgainstGoals'))

In [249]:
df_team_stats_totals.show()

+------+--------------------+------------+-------------+------------+---------------+----------------+------------+-------------+------------+---------------+----------------+---------+-----------+---------+----------------+-----------------+
|Season|                Team|TotalHomeWin|TotalHomeLoss|TotalHomeTie|HomeScoredGoals|HomeAgainstGoals|TotalAwayWin|TotalAwayLoss|TotalAwayTie|AwayScoredGoals|AwayAgainstGoals|TotalWins|TotalLosses|TotalTies|TotalScoredGoals|TotalAgainstGoals|
+------+--------------------+------------+-------------+------------+---------------+----------------+------------+-------------+------------+---------------+----------------+---------+-----------+---------+----------------+-----------------+
|  2016|Athletic Club Bilbao|          12|            3|           4|             36|              19|           6|            9|           3|             19|              27|       18|         12|        7|              55|               46|
|  2017|       Villarreal CF

In [ ]:
# Drop unecessary columns
cols_to_drop = ['TotalHomeWin', 'TotalAwayWin', 'TotalHomeLoss', 'TotalAwayLoss',
                'TotalHomeTie', 'TotalAwayTie', 'HomeScoredGoals', 'AwayScoredGoals',
                'HomeAgainstGoals', 'AwayAgainstGoals']
df_final_stats = df_team_stats_totals.drop(*cols_to_drop)

In [251]:
df_final_stats.show()

+------+--------------------+---------+-----------+---------+----------------+-----------------+
|Season|                Team|TotalWins|TotalLosses|TotalTies|TotalScoredGoals|TotalAgainstGoals|
+------+--------------------+---------+-----------+---------+----------------+-----------------+
|  2016|Athletic Club Bilbao|       18|         12|        7|              55|               46|
|  2017|       Villarreal CF|       19|         12|        8|              55|               42|
|  2015|          Levante UD|        8|         22|        9|              34|               67|
|  2014|          Granada CF|        8|         19|       10|              27|               60|
|  2017|       RC Celta Vigo|       13|         20|        6|              58|               63|
|  2018|        FC Barcelona|       25|          3|       10|             102|               41|
|  2014|          Levante UD|       10|         14|       13|              30|               48|
|  2015|       UD Las Palmas| 

In [257]:
# Create additional columns for goal difference and points
df_processed = df_final_stats.withColumn('GoalDifference', F.col('TotalScoredGoals') - F.col('TotalAgainstGoals')) \
    .withColumn('Points', F.col('TotalWins') * 3 + F.col('TotalTies') * 1) \
    .withColumn('WinPercentage', F.round(F.col('TotalWins') / (F.col('TotalWins') + F.col('TotalLosses') + F.col('TotalTies')) * 100, 2))

In [259]:
# Preview the final processed DataFrame
df_processed.toPandas()

,Season,Team,TotalWins,TotalLosses,TotalTies,TotalScoredGoals,TotalAgainstGoals,GoalDifference,Points,WinPercentage
0,2016,Athletic Club Bilbao,18,12,7,55,46,9,61,48.65
1,2017,Villarreal CF,19,12,8,55,42,13,65,48.72
2,2015,Levante UD,8,22,9,34,67,-33,33,20.51
3,2014,Granada CF,8,19,10,27,60,-33,34,21.62
4,2017,RC Celta Vigo,13,20,6,58,63,-5,45,33.33
...,...,...,...,...,...,...,...,...,...,...
199,2019,Granada CF,7,8,3,24,25,-1,24,38.89
200,2012,RC Celta Vigo,4,10,3,16,22,-6,15,23.53
201,2018,SD Huesca,1,11,5,15,34,-19,8,5.88
202,2013,Atlético Madrid,25,6,7,74,25,49,82,65.79


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Données de test avec différentes dates
test_data = [
    ("2012-08-15", "Début saison 2012/13"),
    ("2012-12-25", "Milieu saison 2012/13"),
    ("2013-01-15", "Début année 2013 - toujours saison 2012/13"),
    ("2013-05-20", "Fin saison 2012/13"),
    ("2013-08-20", "Début saison 2013/14"),
    ("2014-06-10", "Fin saison 2013/14")
]

columns = ["date", "description"]
df_test = spark.createDataFrame(test_data, columns)

# Appliquer la logique des saisons
df_result = (df_test
             .withColumn("date_formatted", F.to_date(F.col("date"), "yyyy-MM-dd"))
             .withColumn("Season",
                         F.when(F.month(F.col("date_formatted")) >= 8,
                                F.concat(F.year(F.col("date_formatted")), F.lit("/"), (F.year(F.col("date_formatted")) + 1)))
                         .otherwise(
                             F.concat((F.year(F.col("date_formatted")) - 1), F.lit("/"), F.year(F.col("date_formatted"))))
                         )
             )

df_result.show()

+----------+--------------------+--------------+---------+
|      date|         description|date_formatted|   Season|
+----------+--------------------+--------------+---------+
|2012-08-15|Début saison 2012/13|    2012-08-15|2012/2013|
|2012-12-25|Milieu saison 201...|    2012-12-25|2012/2013|
|2013-01-15|Début année 2013 ...|    2013-01-15|2012/2013|
|2013-05-20|  Fin saison 2012/13|    2013-05-20|2012/2013|
|2013-08-20|Début saison 2013/14|    2013-08-20|2013/2014|
|2014-06-10|  Fin saison 2013/14|    2014-06-10|2013/2014|
+----------+--------------------+--------------+---------+

